### Understanding SQL Agents

In [1]:
import os

from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_community.utilities import SQLDatabase
from langchain_huggingface import HuggingFaceEmbeddings

c:\Users\Rupesh\Desktop\learning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Rupesh\AppData\Local\Temp\ipykernel_25008\1816509071.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


In [2]:
load_dotenv(override=True)

groq_api_key = os.getenv('GROQ_API_KEY')

if not groq_api_key:
    raise ValueError('GROQ_API_KEY environment variable is not set.')

temperature = 0.7
max_tokens = 1500
model_name = 'llama-3.1-8b-instant'

llm = ChatGroq(
    model=model_name,
    temperature=temperature,
    max_tokens=max_tokens,
    groq_api_key=groq_api_key
)

In [3]:
db_path = "sqlite:///./chinook.db"
db = SQLDatabase.from_uri(db_path)

print(db.dialect)
print(db.get_usable_table_names())

sqlite
['Album', 'Artist', 'Customer', 'Employee', 'Genre', 'Invoice', 'InvoiceLine', 'MediaType', 'Playlist', 'PlaylistTrack', 'Track']


In [4]:
from langchain_classic.chains import create_sql_query_chain

chain = create_sql_query_chain(
    llm=llm,
    db=db,
)

In [5]:
question = "How many employees are there in the database?"

response = chain.invoke({
    "question": question
})

print("Question:", question)
print("Response:", response)

Question: How many employees are there in the database?
Response: Question: How many employees are there in the database?
SQLQuery: SELECT COUNT(*) FROM "Employee"


In [6]:
question = "which country's customers have spent the most?"

chain = create_sql_query_chain(
    llm=llm,
    db=db,
)

response = chain.invoke({
    "question": question
})

print("Question:", question)
print("Response:", response)

Question: which country's customers have spent the most?
Response: Question: which country's customers have spent the most?
SQLQuery: 
SELECT DISTINCT "Country" FROM "Customer" ORDER BY (SELECT SUM("Total") FROM "Invoice" WHERE "CustomerId" = C."CustomerId") DESC LIMIT 5;


In [7]:
question = "How many employees are there in the database?"

response = chain.invoke({
    "question": question
})

print("Question:", question)
print("Response:", response)

Question: How many employees are there in the database?
Response: Question: How many employees are there in the database?
SQLQuery: SELECT COUNT(*) FROM "Employee"


In [8]:
try:
    sql = response.split("SQLQuery:")[-1].strip().rstrip(";") if "SQLQuery:" in response else response
    result = db.run(sql)
    print("DB result:", result)
except Exception as e:
    print(f"[db.run] SQL execution failed (LLM may have generated invalid SQL): {e}")

DB result: [(8,)]


In [9]:
chain.get_prompts()[0].pretty_print()

You are a SQLite expert. Given an input question, first create a syntactically correct SQLite query to run, then look at the results of the query and return the answer to the input question.
Unless the user specifies in the question a specific number of examples to obtain, query for at most 5 results using the LIMIT clause as per SQLite. You can order the results to return the most informative data in the database.
Never query for all columns from a table. You must query only the columns that are needed to answer the question. Wrap each column name in double quotes (") to denote them as delimited identifiers.
Pay attention to use only the column names you can see in the tables below. Be careful to not query for columns that do not exist. Also, pay attention to which column is in which table.
Pay attention to use date('now') function to get the current date, if the question involves "today".

Use the following format:

Question: Question here
SQLQuery: SQL Query to run
SQLResult: Result

In [10]:
from langchain_community.tools.sql_database.tool import QuerySQLDatabaseTool
from langchain_core.tools import tool


@tool
def parse(query_string: str) -> str:
    """Parse the SQL query string and return a human-readable explanation."""

    # Handle full LLM output like: "Question: ...\nSQLQuery: SELECT ..."
    if "SQLQuery:" in query_string:
        sql = query_string.split("SQLQuery:")[-1].strip()
        return sql.rstrip(";").strip()

    # Fallback: strip a "QUERY: " prefix if present
    splitted_string = query_string.split(":")
    if len(splitted_string) >= 2:
        query = splitted_string[1].strip()
    else:
        query = query_string

    return query.rstrip(";").strip()

In [11]:
print("parse (no prefix):", parse.invoke("""
      SELECT COUNT("EmployeeId") AS EmployeeCount
        FROM Employee
      """))

parse (no prefix): SELECT COUNT("EmployeeId") AS EmployeeCount
        FROM Employee


In [12]:
print("parse (QUERY: prefix):", parse.invoke("""
      QUERY: SELECT COUNT("EmployeeId") AS EmployeeCount
        FROM Employee
      """))

parse (QUERY: prefix): SELECT COUNT("EmployeeId") AS EmployeeCount
        FROM Employee


In [13]:
execute_query = QuerySQLDatabaseTool(db=db)
write_query = create_sql_query_chain(
    llm=llm,
    db=db,
)
chain = write_query | parse | execute_query
question = "How many employees are there in the database?"
response = chain.invoke({
    "question": question
})

print("Question:", question)
print("Response:", response)

Question: How many employees are there in the database?
Response: [(8,)]


In [14]:
question = "which country's customers have spent the most?"

execute_query = QuerySQLDatabaseTool(db=db)
write_query = create_sql_query_chain(
    llm=llm,
    db=db,
)
chain = write_query | parse | execute_query

response = chain.invoke({
    "question": question
})

print("Question:", question)
print("Response:", response)

Question: which country's customers have spent the most?
Response: [('USA',)]


In [15]:
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough

In [16]:
answer_prompt = PromptTemplate.from_template(
    """
        Given the following user question, corresponding SQL Query, and SQL Result, answer the user question in a concise manner.
        
        User Question: {question}
        SQL Query: {query}
        SQL Result: {result}
        Answer: 
    """
)

chain = RunnablePassthrough \
    .assign(query=write_query) \
    .assign(result=itemgetter("query") | parse | execute_query) | \
    answer_prompt | llm | StrOutputParser()

In [17]:
question = "How many employees are there in the database?"

response = chain.invoke({
    "question": question
})

print("Question:", question)
print("Response:", response)

Question: How many employees are there in the database?
Response: There are 8 employees in the database.


In [18]:
question = "which country's customers have spent the most?"

response = chain.invoke({
    "question": question
})

print("Question:", question)
print("Response:", response)

Question: which country's customers have spent the most?
Response: The user's SQL query is incorrect because the SUM function cannot be used in the ORDER BY clause without being applied to a specific column.

To fix this, the query should be rewritten to apply the SUM function to the "Total" column, and then select the country with the maximum sum.

Here's the corrected SQL query:

```sql
SELECT "Country"
FROM "Customer"
GROUP BY "Country"
ORDER BY SUM("Total") DESC LIMIT 1
```

However, this query will return only one row, but SQLite's LIMIT clause will return the first row it encounters, which might not necessarily be the row with the maximum sum if there are multiple rows with the same maximum sum.

If you want to return all countries with the maximum sum, you can use the following query:

```sql
SELECT "Country"
FROM "Customer"
GROUP BY "Country"
ORDER BY SUM("Total") DESC
LIMIT 1 OFFSET 0
```

Or, you can use a subquery to get the maximum sum first and then select the countries wi

In [19]:
import ast
import re


def query_as_list(database, query):
    """
    Execute a SQL query and return the results as a list of dictionaries.
    """
    result = database.run(query)
    result = [el for sub in ast.literal_eval(result) for el in sub if el]
    result = [re.sub(r"\b\d+\b", "", string).strip() for string in result]

    return list(set(result))

artists = query_as_list(db, "SELECT Name FROM Artist")
albums = query_as_list(db, "SELECT Title FROM Album")

In [20]:
artists[5]

'Battlestar Galactica (Classic)'

In [21]:
albums[:5]

['A Real Dead One',
 'Angel Dust',
 'O Samba Poconé',
 'Misplaced Childhood',
 "Schubert: The Late String Quartets & String Quintet ( CD's)"]

In [22]:
from langchain_classic.agents.agent_toolkits import create_retriever_tool
from langchain_community.vectorstores import FAISS

In [23]:
vector_database = FAISS.from_texts(
    texts=artists + albums,
    embedding=HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2'),
)

retriever = vector_database.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 2}
)

description = '''
    use to lookup values to filter on.
    input is an approximate spelling of the valid and proper nouns.
    Use the noun most similar to the input.
    If the input is not a valid noun, return an empty string.
'''

retriever_tool = create_retriever_tool(
    retriever=retriever,
    name='retriever',
    description=description,
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4642.56it/s]


In [24]:
response = retriever_tool.invoke("Alis Chains")

print("Input: Alis Chains")
print("Response:", response)

Input: Alis Chains
Response: Alice In Chains

Voodoo Lounge


In [25]:
response = retriever_tool.invoke("Do we have any artists by named Alis Chains")

print("Input: Alis Chains")
print("Response:", response)

Input: Alis Chains
Response: Various Artists

Alice In Chains
